# Running the NLP Project Code on Google Colab


## If running this code on Google Colab:
- Make sure you have the NLP-Final-Project repository saved to your Google Drive already as a .zip file

In [ ]:
import os
import sys

# download miniforge (anaconda alternative --> needed due to issues with Anaconda installation on colab)
if not os.path.exists('/content/miniforge'):
    print("Downloading Miniforge...")
    !wget "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh" -O /content/miniforge.sh
    print("Installing Miniforge...")
    !bash /content/miniforge.sh -b -p /content/miniforge
else:
    print("miniforge already installed.")

conda_bin = "/content/miniforge/bin/conda"
py_env_path = "/content/my_env"
python_exec = f"{py_env_path}/bin/python"
pip_exec = f"{py_env_path}/bin/pip"

# create python 3.8 environment specified by Talk2Car
if not os.path.exists(py_env_path):
    print(f"creating python 3.8 environment...")
    !{conda_bin} create -p {py_env_path} python=3.8 -c conda-forge -y
else:
    print("python 3.8 environment already exists")

if not os.path.exists(pip_exec):
    raise RuntimeError("environment creation failed")

# dependencies
print("\n installing PyTorch 1.8.1 (CUDA 11.1)...")
!{pip_exec} install torch==1.8.1+cu111 torchvision==0.9.1+cu111 -f https://download.pytorch.org/whl/torch_stable.html -q

print(" installing setup tools...")
!{pip_exec} install setuptools==58.0.0 wheel "cython<3.0" numpy==1.24.3 -q

print("installing Spacy 2.2.4 & NuScenes...")
!{pip_exec} install spacy==2.2.4 nuscenes-devkit==1.1.11 scikit-learn==1.3.2 pillow==10.4.0 matplotlib==3.5.3 -q

print("downloading Spacy Model...")
!{python_exec} -m spacy download en_core_web_sm

print("\n setup completed")
print(f" For running scripts: {python_exec}")

In [ ]:
# setting up the NLP-Final-Project repo in colab
import os
import shutil
import time
import sys
from google.colab import drive


DRIVE_ZIP_PATH = "" # Path to your zip file in google drive
LOCAL_EXTRACT_PATH = '/content/local_data'

start_time = time.time()
print("Initializing setup:")

# mount drive
os.chdir('/content')
if not os.path.exists('/content/drive'):
    print("mounting drive...")
    drive.mount('/content/drive')

# checking if data already exists locally on Colab
if os.path.exists(LOCAL_EXTRACT_PATH):
    print("lsocal data folder found.")
else:
    print("local data not found. unzipping from drive now...")
    if not os.path.exists('/content/temp.zip'):
        shutil.copy(DRIVE_ZIP_PATH, '/content/temp.zip')
    os.makedirs(LOCAL_EXTRACT_PATH)
    !unzip -q /content/temp.zip -d {LOCAL_EXTRACT_PATH}


In [ ]:
# Model training
import os
import shutil
import time
import sys
from google.colab import runtime

LOCAL_EXTRACT_PATH = '/content/local_data'
found_train = False
baseline_dir = ""
project_root = ""

for root, dirs, files in os.walk(LOCAL_EXTRACT_PATH):
    if 'train.py' in files and 'baseline' in root:
        baseline_dir = root
        project_root = os.path.dirname(baseline_dir)
        found_train = True
        break

data_dir = os.path.join(project_root, 'data')
os.chdir(baseline_dir)

sys.path.append(project_root)
env_site_packages = "/content/my_env/lib/python3.8/site-packages"
os.environ['PYTHONPATH'] = f"{project_root}:{baseline_dir}:{env_site_packages}"

print("Output will be saved to model_output.txt")

try:
    # run the training script 
    !/content/my_env/bin/python -u train.py \
        --root "{data_dir}" \
        --lr 0.001 \
        --nesterov \
        --evaluate \
        --batch-size 12 \
        --workers 4 \
        --print-freq 50 | tee model_output.txt

    print("training done")

    drive_backup_dir =  "" # the location of where you want to save the training output to 
    if not os.path.exists(drive_backup_dir):
        os.makedirs(drive_backup_dir)

    files_to_save = ['best_model.pth.tar', 'checkpoint.pth.tar', 'model_output.txt']

    for filename in files_to_save:
        if os.path.exists(filename):
            if filename == 'model_output.txt':
                timestamp = time.strftime("%Y%m%d-%H%M%S")
                dest_name = f"log_{timestamp}.txt"
                shutil.copy(filename, os.path.join(drive_backup_dir, dest_name))
                print(f"   -> Saved Log as: {dest_name}")
            else:
                shutil.copy(filename, os.path.join(drive_backup_dir, filename))
                print(f"   -> Saved Model: {filename}")

    print("output saved")

except Exception as e:
    print(f"Error during execution: {e}")
    print("Runtime kept alive for debugging.")

In [ ]:
# disconnect model to prevent idle GPU credit use
time.sleep(5)
runtime.unassign()
print("disconnected runtime")